In [0]:
%run "/Workspace/Users/vtilakumara@academic.mit.edu.au/config"

In [0]:
# if the job parameter is not set, default to dev
try:
    ENV = dbutils.widgets.get("ENV")
except:
    ENV = "dev"

print(f"Starting pipeline for {ENV}")
print(f"Environment is {ENV}")
print(f"Starting Task 1 - Bronze layer creation for price and demand")

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql import functions as F

In [0]:
landing_path = "abfss://enerygyau2026@energyau2026vtk.dfs.core.windows.net/landing/price_and_demand/"
checkpoint_path = "abfss://enerygyau2026@energyau2026vtk.dfs.core.windows.net/checkpoints/price_and_demand/"
schema_path = "abfss://enerygyau2026@energyau2026vtk.dfs.core.windows.net/schemas/price_and_demand/"

In [0]:
schema = StructType([
    StructField("REGION", StringType(), True),
    StructField("SETTLEMENTDATE", StringType(), True),
    StructField("TOTALDEMAND", DoubleType(), True),
    StructField("RRP", DoubleType(), True),
    StructField("PERIODTYPE", StringType(), True)
])

In [0]:
df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.useNotifications", "false")   # use directory listing
        .option("header", "true")
        .schema(schema)
        .load(landing_path)
)

print(df)

In [0]:
df = (
    df
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingested_at", F.current_timestamp())
)

In [0]:
# Writes the streaming DataFrame 'df' to a Delta table.
# - Uses 'checkpoint_path' for checkpointing.
# - 'availableNow=True' triggers batch processing of all available data.
# - 'mergeSchema' allows schema evolution.
# - Writes to the table specified by 'BRONZE_TABLE'.
(
    df.writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .trigger(availableNow=True)
      .option("mergeSchema", "true")
      .toTable(BRONZE_TABLE)
)

In [0]:
# %sql
# SELECT count(*) 
# FROM energyau2026.bronze.price_and_demand;
display(spark.sql(f"SELECT count(*) FROM {BRONZE_TABLE}"))

In [0]:
%sql
SELECT *
FROM energyau2026.energyau_dev_bronze.price_and_demand
LIMIT 10;